vielä kerran: henkilöittäin, 2015-2025, varianssianalyysi per hlö
- ympätään datat 2015-2025
- erotellaan henkilöittäin
- pidetään puheiden aikaleimat, paino pois vuosilta
- lemmatisoitu ja lemmatisoimaton
- varianssi ajan myötä, tarkastelu per henkilö
- koitetaan saada ennen ennen kesälomia

In [1]:
# import packages, declare constants, get the parent directory
import pandas as pd
import numpy as np
import re
import os
import time
#import decimal
from scipy import stats
#from wordcloud import WordCloud
import matplotlib.pyplot as plt
from matplotlib import colormaps
#from .utils import helpers
#import simplemma
import pyvoikko

CHATGPT_RELEASE_YEAR = int(2022)
FINNISH_ALPHABET = 'abcdefghijklmnopqrstuvwxyzåäö'

def get_parent_directory() -> str:
    """Get the parent directory for handling csv files.

    Returns:
        string: the path to the directory where directories for csv files are located
    """
    #create relative path for parent
    relative_parent = os.path.join(os.getcwd(), '..')

    #use abspath for absolute parent path
    return str(os.path.abspath(relative_parent)).replace('\\', '/')

directory = get_parent_directory()

In [2]:
# Read the csvs of speeches, years 2015-2025. Combine into a singel dataframe. The dataframe will include a column with lemmatised version of the speech 
# and an original unlemmatised version.
# Running this cell does nothing but creates the dataset for analysis.
#for csv in [i for i in os.listdir(f'{directory}/csv_lemmatized/') if re.search('\d+',i)[0] >= 2015]:
df_all_years = pd.DataFrame()

for csv in [i for i in os.listdir(f'{directory}/csv_lemmatized/') if re.search(r'\d+', i)]:
    if int(re.search(r'\d+', csv)[0]) < 2015:
        pass
    else:
        df_year = pd.read_csv(f'{directory}/csv_lemmatized/{csv}', sep=';', header=0, encoding='utf-8')
        df_all_years = pd.concat([df_all_years, df_year])


In [3]:
# Keep only rows where the speaker identifier (speaker_id) is present. Some are missing for some reason.
df_all_years = df_all_years.loc[pd.notna(df_all_years['speaker_id'])]
df_all_years = df_all_years.loc[df_all_years['speaker_id'].str.strip().str.len()>0]

In [4]:
# Keep only rows with the speech_type category in: 'Esittelypuheenvuoro', 'Ryhmäpuheenvuoro', 'Varsinainen puheenvuoro', 'Puheenvuoro'
#    -> Drop rows with other categories.
#    This is to limit the rows to speeches where preparation (ergo option for using generative AI) is necessary.
# Clean and prep data before filtering:
# - Clean speech_type, fix known spelling errors

list_wanted_speech_types = ['Esittelypuheenvuoro', 'Ryhmäpuheenvuoro', 'Varsinainen puheenvuoro', 'Puheenvuoro']

def clean_speech_type(speech_type: str) -> str:
    if 'vastauspuheenvuoro' in speech_type:
        return 'Vastauspuheenvuoro'
    elif 'esittelypuheenvuoro' in speech_type:
        return 'Esittelypuheenvuoro'
    else:
        return speech_type.strip()

# Clean speech_type
df_all_years.loc[:, 'speech_type'] = df_all_years['speech_type'].apply(clean_speech_type)
# Filter rows
df_all_years = df_all_years[df_all_years['speech_type'].isin(list_wanted_speech_types)]

In [5]:
# What happens in this cell:
# Data preparation:
# - Lemmatise the rows which have not yet been lemmatised (due to data errors or some coding oversight earlier in the process)
# - The actual process of lemmatisation is done with separate functions
# - Because lemmatisation takes long, the process is split into parts
# -- filter rows which have to be lemmatised -> store in a separate df
# -- lemmatise these rows in their respective df
# -- concat these rows back into the main df

def clean_string(string: str) -> str:
    try:
        # remove blanks in start and end
        string = string.strip()
        string = string.lower()
        # the string must contain characters
        if any(c in string for c in FINNISH_ALPHABET)==False:
            string = ''
        # remove tabulations, line breaks etc., also special characters
        remove_these = r'[\+\*!"”’?.:,…()§\'[\] \t\n\r\f\v]'
        string = re.sub(remove_these, '', string)
        # remove weird parentheses and backwards linebreaks from starts of strings
        string = re.sub(r'^\)\\[a-z]', '', string)
        # remove weird '\[alphabet]' strings at start of strings
        string = re.sub(r'^\\[a-z]', '', string)
        # remove numbers
        string = re.sub(r'[0-9]', '', string)
        # remove dashes '-' at the start and end of string
        string = re.sub(r'^-|-$', '', string)
        # remove individual forward and backward slashes '/', '\'
        string = re.sub(r'[\/\\]', '', string)
        # remove double dashes '--'
        string = string.replace('--', '-')
        # remove the equal sign '='
        string = string.replace('=', '')
        # at the end of the cleaning, remove all characters from the string which are not in the alphabet except for dash (compound words)
        remove_these = ''.join([str(c) for c in string if c != '-' and c not in [i for i in FINNISH_ALPHABET]])
        string = re.sub(remove_these, '', string)
        # remove blanks in start and end again
        string = string.strip()
        # remove empty if string length < 2
        string = '' if len(string) < 2 else string
        return string
    except:
        print(f'Unexpected error at helpers.clean_string(), string: {string}')
        raise

def pyvoikko_wrapper(str_in: str) -> str:
    """A small wrapper for pyvoikko Finnish text analysis tool.
    Used to help lemmatizing strings. Uses the clean_string helper function.
    NOTE: omits the etunimi (given name) word class returned in PyVoikko.

    Args:
        str_in (str): input string to lemmatize

    Returns:
        str: string of lemmatized words; input string in lemmatized form
    """
    # format output string
    str_out = str()
    # turn input string into a list
    str_in = str_in.split()
    for i in str_in:
        i = pyvoikko.analyse(clean_string(i))
        if (i is not None) and (len(i)>0) and (i[0].CLASS != 'etunimi'):
            str_out = str_out+' '+i[0].BASEFORM
    if len(str_out) == 0:
        return None
    else:
        return str_out.strip()

lemmatise_df = df_all_years.loc[(pd.isna(df_all_years['content_lemmatized']))&(df_all_years['lang']=='fi')&(type(df_all_years['content'])==str)]

if len(lemmatise_df)>0:
    # lemmatise material, if "lemmatisable" rows exist
    lemmatise_df.loc[:, 'content_lemmatized'] = lemmatise_df.apply(lambda x: pyvoikko_wrapper(x['content']), axis=1)
    # update df_all_years dataframe's content_lemmatized with content created above
    df_all_years.update(lemmatise_df)

In [6]:
# Keep only rows where content_lemmatized is not None and the length of content_lemmatized is over 0 
# -> actual information is stored in there
df_all_years = df_all_years.loc[(pd.notna(df_all_years['content_lemmatized']))&(len(df_all_years['content_lemmatized'])>0)]

In [8]:
# Clean lemmatised content (content_lemmatized column). Drop special characters etc., so that the lemmatised content is similar throughout the dataframe
# and special characters etc. do not interfere.
# A small wrapper for the clean_string function for this purpose.

def clean_string_wrapper(str_in: str) -> str:
    # format output string
    str_out = str()
    # turn input string into a list of words, cut at SPACE
    str_in = str_in.split()
    for i in str_in:
        i = clean_string(i)
        if (i is not None) and (len(i)>0):
            str_out = str_out+' '+i
    if len(str_out)==0:
        return None
    else:
        return str_out.strip()

df_all_years.loc[:, 'content_lemmatized'] = df_all_years.apply(lambda x: clean_string_wrapper(x['content_lemmatized']), axis=1)

# If lemmatisation and string cleaning has nulled any speeches, drop such rows
df_all_years = df_all_years.loc[pd.notna(df_all_years['content_lemmatized'])]
df_all_years.reset_index(inplace=True)

In [9]:
# Data preparation is now finished.
# What the dataframe looks like now:
df_all_years[:1].T

,0
index,24
speech_id,2015_2_6
session,2/2015
date,2015-05-05
start_time,14.01
end_time,14.15
given,Sirkka-Liisa
family,Anttila
role,Kansanedustaja
party,KESK


The analysis part starts here.
Next the goal is to calculate word frequencies per speaker to see if there are significant changes speaker-wise.
The data will not be grouped by year, so the frequencies will not be based on yearly aggregates of observations. Frequencies will be calculated for each speech. Each speech has a unique speech identifier (speech_id).
To make it possible to do this for each speaker, several different datasets will be created:
- A list that holds all unique speaker identifiers.
- A dataframe that holds speaker_id, speech_id and session identifier (session)
- A dataframe that holds all words and their frequencies per speech

In [10]:
# Next:
# - Store each speaker_id (speaker identifier) in a list: speakers
speakers = df_all_years['speaker_id'].unique()

In [11]:
# Next: Get each speakers' speech and session identifiers. Store them in a dataframe: df_speeches_sessions
df_speeches_sessions = pd.DataFrame(columns=['speaker_id','session','speech_id','content','content_lemmatized']).astype({'speaker_id':str, 'session':str, 'speech_id':str, 'content':str, 'content_lemmatized':str})

# Populate the dataframe
for speaker in speakers:
    df_speeches_sessions = pd.concat([df_speeches_sessions, df_all_years[['speaker_id','session','speech_id','content','content_lemmatized']].loc[df_all_years['speaker_id']==speaker]], axis=0, ignore_index=True)

In [15]:
# Next: Get all words per speaker per speech. Store them in a dataframe: df_speech_session_words.
# Store the dataframe (containing information 'speaker_id','session','speech_id','word','word_n') as a csv:
#    a csv will be created for each speaker; each speaker's data will be in their respective csv.
def count_word_freqs_in_string(string: str):
    """Counts the words in the input string.
    Returns a dictionary where the word is the key and the frequency is the value.
    """
    if ((string is None) or (string == 'nan')):
        return None
    else:
        words_list = re.split(' ', string)
        wordfreq_dict = {}
        for word in words_list:
            if word not in wordfreq_dict.keys():
                wordfreq_dict[word] = 1
            else:
                wordfreq_dict[word] += 1

        return wordfreq_dict

# Populate the dataframe with help from the function declared above
for speaker in speakers:
    # Format a df to store the results -> this will be saved as a csv for later use
    df_speech_session_words = pd.DataFrame(columns=['speaker_id','session','speech_id','word','word_n']).astype({'speaker_id':str,'session':str,'speech_id':str,'word':str,'word_n':int})
    # A name for csv in a helper variable.
    csv_name = f'speaker_{speaker}_words_sessions.csv'
    csv_save_directory = f'{directory}/csv_analysis/person_variance/'
    csv_save_path = f'{directory}/csv_analysis/person_variance/{csv_name}'
    # If the csv already exists (speaker's data has already been created) -> skip and move to the next speaker
    if csv_name in os.listdir(csv_save_directory):
        pass
    else:
        for speech in df_speeches_sessions['speech_id'].loc[df_speeches_sessions['speaker_id']==speaker]:
            session = df_speeches_sessions['session'].loc[(df_speeches_sessions['speaker_id']==speaker)&(df_speeches_sessions['speech_id']==speech)].item()
            cont_lem = df_speeches_sessions['content_lemmatized'].loc[(df_speeches_sessions['speaker_id']==speaker)&(df_speeches_sessions['speech_id']==speech)].item()
            # We have now targeted down to the speech content. Let's count how many times each word appears in the speech.
            word_counts = count_word_freqs_in_string(cont_lem)
            # Let's store the results!
            for word, count in word_counts.items():
                df_speech_session_words = pd.concat([df_speech_session_words, pd.DataFrame.from_dict(data={'speaker_id':[speaker],'session':[session],'speech_id':[speech],'word':[word],'word_n':[count]}, orient='columns')],axis=0, ignore_index=True)
        # Store the speaker's dataframe as a csv
        try:
            df_speech_session_words.to_csv(csv_save_path, sep=';', header=True, index=False, encoding='utf-8')
        except FileExistsError:
            os.remove(csv_save_path)
            df_speech_session_words.to_csv(csv_save_path, sep=';', header=True, index=False, encoding='utf-8')

Data preparation at the individual (speaker) level is now finished.

In [49]:
#for csv in [f for f in os.listdir(f'{directory}/csv_analysis/person_variance/') if re.search(r'speaker_.+\.csv$', f)]:
#    print(csv)
df = pd.read_csv(f'{directory}/csv_analysis/person_variance/speaker_301_words_sessions.csv', sep=';', header=0, encoding='utf-8')

In [ ]:
# Check if sessions _before_ and _after_ CHATGPT_RELSEASE_YEAR (2022) exist.
# If not, pass the speaker and move on to the next.
# First, create a dataframe to store the results.
speaker_word_anova = pd.DataFrame(columns=['speaker_id','word','statistic','pvalue']).astype({'speaker_id':str,'word':str,'statistic':float,'pvalue':float})
if (len([i for i in df['session'].unique() if int(i[-4:])<CHATGPT_RELEASE_YEAR])>0) & (len([i for i in df['session'].unique() if int(i[-4:])>CHATGPT_RELEASE_YEAR])>0):
    # Sessions exists.
    # Next: get all the unique words, store in a list: words
    words = df['word'].unique()
    speaker = df['speaker_id'].unique()[0].astype(str)
    # Loop through the words, get the word counts for sessions before and after CHATGPT_RELEASE_YEAR and store in respective lists
    for word in words:
        counts_before = df['word_n'].loc[(df['word']==word)&(df['session'].isin([i for i in df['session'].unique() if int(i[-4:])<=CHATGPT_RELEASE_YEAR]))].tolist()
        counts_after = df['word_n'].loc[(df['word']==word)&(df['session'].isin([i for i in df['session'].unique() if int(i[-4:])>CHATGPT_RELEASE_YEAR]))].tolist()
        # Run ANOVA if items exist in both lists
        if (len(counts_before)>0) & (len(counts_after)>0):
            #print(f'word: {word}, counts_before: {len(counts_before)}, counts_after: {len(counts_after)}')
            try:
                statistic, pvalue = stats.f_oneway(counts_before, counts_after, axis=0, equal_var=True, nan_policy='propagate', keepdims=False)
                #print(f'statistic: {statistic}, pvalue: {pvalue}')
                speaker_word_anova = pd.concat([speaker_word_anova, pd.DataFrame.from_dict(data={'speaker_id':[speaker],'word':[word],'statistic':[statistic],'pvalue':[pvalue]}, orient='columns')], axis=0, ignore_index=True)
            except SmallSampleWarning:
                # if the sample is too small (too few observations) -> do nothing
                pass


In [62]:
speaker_word_anova[speaker_word_anova['pvalue']<0.05]

,speaker_id,word,statistic,pvalue
0,301,arvoisa,12.163781,5.345096e-04
2,301,puhemies,11.045750,9.564335e-04
4,301,kertomus,6.072704,2.402326e-02
11,301,tilanne,3.979847,4.729877e-02
12,301,ja,34.344597,8.652686e-09
...,...,...,...,...
2456,301,rikkomusmenettely,inf,0.000000e+00
2466,301,lakko-oikeus,inf,0.000000e+00
2468,301,osapuoli,inf,0.000000e+00
2476,301,matias,inf,0.000000e+00


In [ ]:
def get_normalised(df: pd.DataFrame, speaker_id: str, year: int, word_n: int) -> float:
    w_min = df['word_n'].loc[(df['speaker_id']==speaker_id)&(df['year']==year)].min()
    w_max = df['word_n'].loc[(df['speaker_id']==speaker_id)&(df['year']==year)].max()
    return ((word_n-w_min)/(w_max-w_min))

In [58]:
#[i for i in df['session'].unique() if int(i[-4:])>CHATGPT_RELEASE_YEAR]
word='arvoisa'
counts_before = df['word_n'].loc[(df['word']==word)&(df['session'].isin([i for i in df['session'].unique() if int(i[-4:])<=CHATGPT_RELEASE_YEAR]))].tolist()
counts_after = df['word_n'].loc[(df['word']==word)&(df['session'].isin([i for i in df['session'].unique() if int(i[-4:])>CHATGPT_RELEASE_YEAR]))].tolist()
if (len(counts_before)>0) & (len(counts_after)>0):
    print(f'word: {word}, counts_before: {len(counts_before)}, counts_after: {len(counts_after)}')

word: arvoisa, counts_before: 401, counts_after: 57


In [48]:
#[i for i in df['session'].unique() if int(i[-4:])<=CHATGPT_RELEASE_YEAR]
df['speaker_id'].unique()[0].astype(str)

np.str_('1031')